In [17]:
#Imports
import torch
from data_loading import load_data
from basic_multihead_transformer import MultiheadAttentionNN
import torch.optim as optim

In [18]:
#Load in data
#Data_size is smaller than the full 60000 for runtime considerations
batch_size = 32
data_size = 60000
tts_ratio = 0.8
dataloader_train, dataloader_test = load_data(batch_size, data_size, tts_ratio=tts_ratio)
test_size = len(dataloader_test)*batch_size
train_size = len(dataloader_train)*batch_size

In [19]:
#Define model architecture and optimizer parameters
model = MultiheadAttentionNN(
    batch_size=batch_size,
    patch_size=14,
    output_dim=10,
    model_dim=64,
    key_dim=64,
    value_dim=64,
    heads=8
)
lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.CrossEntropyLoss()

In [20]:
def train_epoch(epoch):
    running_loss = 0
    running_acc = 0
    for i, data in enumerate(dataloader_train):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = model(inputs)

        CLS = outputs[:, 0, :]

        with torch.no_grad():
            preds = torch.argmax(CLS, dim=1)
            correct = (preds == labels)
            accuracy = correct.float().sum()
            running_acc += accuracy


        loss = loss_fn(CLS, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
    
    return running_loss, running_acc

In [21]:
#Train the model
model.train()
for epoch in range(50):
    epoch_loss, accuracy = train_epoch(epoch)
    print(f"Epoch {epoch+1} | Loss: {epoch_loss/(train_size//batch_size):.4f} | Accuracy: {accuracy/train_size:.4f}")

Epoch 1 | Loss: 0.7776 | Accuracy: 0.7798
Epoch 2 | Loss: 0.2875 | Accuracy: 0.9143
Epoch 3 | Loss: 0.2136 | Accuracy: 0.9354
Epoch 4 | Loss: 0.1704 | Accuracy: 0.9479
Epoch 5 | Loss: 0.1433 | Accuracy: 0.9561
Epoch 6 | Loss: 0.1282 | Accuracy: 0.9607
Epoch 7 | Loss: 0.1082 | Accuracy: 0.9660
Epoch 8 | Loss: 0.0992 | Accuracy: 0.9688
Epoch 9 | Loss: 0.0887 | Accuracy: 0.9726
Epoch 10 | Loss: 0.0846 | Accuracy: 0.9741
Epoch 11 | Loss: 0.0783 | Accuracy: 0.9758
Epoch 12 | Loss: 0.0703 | Accuracy: 0.9775
Epoch 13 | Loss: 0.0658 | Accuracy: 0.9791
Epoch 14 | Loss: 0.0608 | Accuracy: 0.9809
Epoch 15 | Loss: 0.0638 | Accuracy: 0.9804
Epoch 16 | Loss: 0.0544 | Accuracy: 0.9828
Epoch 17 | Loss: 0.0565 | Accuracy: 0.9825
Epoch 18 | Loss: 0.0516 | Accuracy: 0.9837
Epoch 19 | Loss: 0.0495 | Accuracy: 0.9850
Epoch 20 | Loss: 0.0428 | Accuracy: 0.9870
Epoch 21 | Loss: 0.0459 | Accuracy: 0.9858
Epoch 22 | Loss: 0.0422 | Accuracy: 0.9870
Epoch 23 | Loss: 0.0489 | Accuracy: 0.9851
Epoch 24 | Loss: 0.0

In [22]:
def test_model():
    running_acc = 0
    for i, data in enumerate(dataloader_test):
        inputs, labels = data

        outputs = model(inputs)

        CLS = outputs[:, 0, :]

        with torch.no_grad():
            preds = torch.argmax(CLS, dim=1)
            correct = (preds == labels)
            accuracy = correct.float().sum()
            running_acc += accuracy

    return running_acc

test_accuracy = test_model()
test_size = len(dataloader_test)*batch_size
print(f"Test accuracy: {test_accuracy/test_size:.4f}")

Test accuracy: 0.9679
